# Notebook 02 — CSV (landing-zone) → Delta Lake (bronze)

Lê os CSVs do bucket `landing-zone` e converte para **Delta Lake** no bucket `bronze`.

```
MinIO / landing-zone / tabela /*.csv
        │
        ▼  PySpark + delta-spark
MinIO / bronze / tabela /  (Delta Table — arquivos Parquet + _delta_log/)
```

### O que é Delta Lake?
| Recurso | Descrição |
|---------|----------|
| **ACID** | Transações atômicas, consistentes, isoladas, duráveis |
| **Time Travel** | Consultar versões anteriores com `versionAsOf` |
| **Schema enforcement** | Rejeita dados com schema incompatível |
| **DML** | INSERT / UPDATE / DELETE nativos |
| **Audit log** | `_delta_log/` registra cada operação |

In [ ]:
import os
import boto3
from dotenv import load_dotenv
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from delta import configure_spark_with_delta_pip

load_dotenv()

MINIO_ENDPOINT   = os.getenv('MINIO_ENDPOINT',   'http://localhost:9020')
MINIO_ACCESS_KEY = os.getenv('MINIO_ACCESS_KEY', 'minioadmin')
MINIO_SECRET_KEY = os.getenv('MINIO_SECRET_KEY', 'minioadmin')
BUCKET_LANDING   = os.getenv('MINIO_BUCKET_LANDING', 'landing-zone')
BUCKET_BRONZE    = os.getenv('MINIO_BUCKET_BRONZE',  'bronze')

print(f'Landing : s3a://{BUCKET_LANDING}')
print(f'Bronze  : s3a://{BUCKET_BRONZE}')


## 1. SparkSession com Delta Lake + MinIO

In [ ]:
# Lista de pacotes extras necessários para o S3/MinIO
EXTRA_PACKAGES = [
    "org.apache.hadoop:hadoop-aws:3.3.4",
    "com.amazonaws:aws-java-sdk-bundle:1.12.367"
]

builder = (
    SparkSession.builder
    .appName('DespesaPublica-02-CSV-to-Delta')
    .config('spark.sql.extensions',
            'io.delta.sql.DeltaSparkSessionExtension')
    .config('spark.sql.catalog.spark_catalog',
            'org.apache.spark.sql.delta.catalog.DeltaCatalog')
    .config('spark.hadoop.fs.s3a.endpoint',          MINIO_ENDPOINT)
    .config('spark.hadoop.fs.s3a.access.key',        MINIO_ACCESS_KEY)
    .config('spark.hadoop.fs.s3a.secret.key',        MINIO_SECRET_KEY)
    .config('spark.hadoop.fs.s3a.path.style.access', 'true')
    .config('spark.hadoop.fs.s3a.impl',
            'org.apache.hadoop.fs.s3a.S3AFileSystem')
    .config('spark.hadoop.fs.s3a.connection.ssl.enabled', 'false') # Evita erro de SSL no MinIO local
)

# Inicializa o Spark injetando os pacotes extras diretamente pelo helper do Delta
spark = configure_spark_with_delta_pip(builder, extra_packages=EXTRA_PACKAGES).getOrCreate()

spark.sparkContext.setLogLevel('WARN')
print(f'Spark {spark.version} com Delta Lake pronto!')

## 2. Criar bucket `bronze`

In [ ]:
s3 = boto3.client(
    's3',
    endpoint_url=MINIO_ENDPOINT,
    aws_access_key_id=MINIO_ACCESS_KEY,
    aws_secret_access_key=MINIO_SECRET_KEY
)

existing = [b['Name'] for b in s3.list_buckets()['Buckets']]
if BUCKET_BRONZE not in existing:
    s3.create_bucket(Bucket=BUCKET_BRONZE)
    print(f"Bucket '{BUCKET_BRONZE}' criado!")
else:
    print(f"Bucket '{BUCKET_BRONZE}' ja existe.")


## 3. Conversão CSV → Delta Lake

Cada tabela é lida do `landing-zone`, tem seus tipos ajustados e é gravada como **Delta Table não gerenciada** no `bronze`.

> **Tabela não gerenciada**: dados ficam no MinIO. O Spark gerencia apenas os metadados. Um `DROP TABLE` remove só o registro do catálogo, **não** os dados físicos.

In [ ]:
# Mapeamento de tipos por tabela
SCHEMA = {
    'orgao':          {'id_orgao':'int','codigo_orgao':'string','nome_orgao':'string','sigla':'string','ativo':'int'},
    'unidade':        {'id_unidade':'int','id_orgao':'int','codigo_unidade':'string','nome_unidade':'string','sigla':'string'},
    'programa':       {'id_programa':'int','codigo_programa':'string','nome_programa':'string','ano_inicio':'int','ativo':'int'},
    'acao':           {'id_acao':'int','id_programa':'int','codigo_acao':'string','nome_acao':'string'},
    'fonte_recurso':  {'id_fonte':'int','codigo_fonte':'string','descricao':'string'},
    'credor':         {'id_credor':'int','cnpj':'string','razao_social':'string','municipio':'string','uf':'string','ativo':'int'},
    'empenho':        {'id_empenho':'int','numero_empenho':'string','id_unidade':'int','id_acao':'int',
                       'id_fonte':'int','id_credor':'int','data_empenho':'date','valor_empenho':'double',
                       'tipo_empenho':'string','modalidade_licitacao':'string','situacao':'string','descricao':'string'},
    'item_empenho':   {'id_item':'int','id_empenho':'int','descricao_item':'string','quantidade':'int',
                       'valor_unitario':'double','valor_total':'double'},
    'liquidacao':     {'id_liquidacao':'int','id_empenho':'int','numero_liquidacao':'string',
                       'data_liquidacao':'date','valor_liquidado':'double','situacao':'string'},
    'pagamento':      {'id_pagamento':'int','id_liquidacao':'int','id_credor':'int',
                       'numero_ordem_bancaria':'string','data_pagamento':'date',
                       'valor_pago':'double','banco':'string','agencia':'string','situacao':'string'},
    'historico_preco':{'id_historico':'int','descricao_item':'string','ano_referencia':'int',
                       'valor_medio':'double','quantidade_pesquisada':'int','fonte_pesquisa':'string'},
}

print('=== CONVERSAO: CSV -> Delta Lake ===\n')

for tabela, tipos in SCHEMA.items():
    # 1. Ler CSV do landing-zone
    df = (
        spark.read
        .option('header', 'true')
        .option('inferSchema', 'true')
        .csv(f's3a://{BUCKET_LANDING}/{tabela}')
    )

    # 2. Aplicar tipos corretos
    for col, tipo in tipos.items():
        if col in df.columns:
            df = df.withColumn(col, F.col(col).cast(tipo))

    # 3. Adicionar colunas de auditoria
    df = (
        df
        .withColumn('dt_carga', F.current_timestamp())
        .withColumn('origem',   F.lit('sqlserver -> landing-zone -> bronze'))
    )

    # 4. Gravar como Delta (TABELA NAO GERENCIADA — path no MinIO)
    delta_path = f's3a://{BUCKET_BRONZE}/{tabela}'
    df.write.format('delta').mode('overwrite').save(delta_path)

    print(f'  {tabela:20s} {df.count():4d} registros -> {delta_path}')

print('\nConversao para Delta Lake concluida!')


## 4. Validação das Delta Tables

In [ ]:
from delta.tables import DeltaTable

print('=== VALIDACAO DELTA TABLES ===\n')
print(f'  {"tabela":<20} {"delta?":<8} {"versao":<8} {"registros"}')
print('  ' + '-'*45)

for tabela in SCHEMA:
    path = f's3a://{BUCKET_BRONZE}/{tabela}'
    is_delta = DeltaTable.isDeltaTable(spark, path)
    if is_delta:
        dt  = DeltaTable.forPath(spark, path)
        ver = dt.history(1).select('version').first()[0]
        n   = spark.read.format('delta').load(path).count()
        print(f'  {tabela:<20} {"OK":<8} {ver:<8} {n}')
    else:
        print(f'  {tabela:<20} FALHOU')


## 5. Registrar tabelas no catálogo Spark

> Registrar permite usar `spark.sql('SELECT * FROM despesa_publica.empenho')`.
> Como usamos `LOCATION`, continuam sendo **tabelas não gerenciadas**.

In [ ]:
spark.sql('CREATE DATABASE IF NOT EXISTS despesa_publica')

for tabela in SCHEMA:
    delta_path = f's3a://{BUCKET_BRONZE}/{tabela}'
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS despesa_publica.{tabela}
        USING DELTA
        LOCATION '{delta_path}'
    """)

print('Tabelas registradas no catalogo Spark!')
print('Tipo: NAO GERENCIADAS (dados no MinIO, metadados no catalogo Spark)\n')

# Query de exemplo usando SQL
print('=== Empenhos por Orgao ===')
spark.sql("""
    SELECT o.sigla, o.nome_orgao,
           COUNT(e.id_empenho)          AS total_empenhos,
           ROUND(SUM(e.valor_empenho),2) AS valor_total_R
    FROM   despesa_publica.empenho  e
    JOIN   despesa_publica.unidade  u ON e.id_unidade = u.id_unidade
    JOIN   despesa_publica.orgao    o ON u.id_orgao   = o.id_orgao
    GROUP  BY o.sigla, o.nome_orgao
    ORDER  BY valor_total_R DESC
""").show(truncate=50)

spark.stop()
print('Notebook 02 concluido!')
